In [14]:
import pandas as pd
df = pd.read_excel(r'D:\PYTHON\Machine Learning\Data_set\Crop_Yield_Messy_Dataset_500.xlsx')
df = df.drop_duplicates()
df.columns = df.columns.str.strip().str.lower()
#print(df['yield_ton_per_hectare'].mean())

In [15]:
import numpy as np
numeric_cols = df.select_dtypes(include='number').columns
df[numeric_cols] = df[numeric_cols].mask(df[numeric_cols] < 0, np.nan)

In [16]:
df['state'] = df['state'].str.strip().str.lower()

df['district'] = df['district'].str.strip().str.lower()

df['crop'] = df['crop'].str.strip().str.lower()

df['season'] = df['season'].str.strip().str.lower()

In [17]:
from sklearn.model_selection import train_test_split


unnecessary_col = ['yield_ton_per_hectare','district','state','fertilizer_used','irrigation']
X = df.drop(unnecessary_col, axis=1)
y = df['yield_ton_per_hectare']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [18]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')

num_cols = X_train.select_dtypes(include='number').columns
X_train[num_cols] = imputer.fit_transform(X_train[num_cols])
X_test[num_cols] = imputer.transform(X_test[num_cols])

In [ ]:
from sklearn.impute import SimpleImputer

cat_imputer = SimpleImputer(strategy='most_frequent')

categorical_cols = X_train.select_dtypes(include='object').columns
X_train[categorical_cols] = cat_imputer.fit_transform(X_train[categorical_cols])
X_test[categorical_cols] = cat_imputer.transform(X_test[categorical_cols])

In [20]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

cols = ['crop', 'season', 'soil_type']

X_train_encoded = encoder.fit_transform(X_train[cols])
X_test_encoded = encoder.transform(X_test[cols])

In [21]:
X_train_encoded = pd.DataFrame(
    X_train_encoded,
    columns=encoder.get_feature_names_out(cols),
    index=X_train.index
)

X_test_encoded = pd.DataFrame(
    X_test_encoded,
    columns=encoder.get_feature_names_out(cols),
    index=X_test.index
)

In [22]:
X_train = X_train.drop(cols, axis=1)
X_test = X_test.drop(cols, axis=1)

X_train = pd.concat([X_train, X_train_encoded], axis=1)
X_test = pd.concat([X_test, X_test_encoded], axis=1)

In [23]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [24]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
print('Based on test data : ',y_pred[0])

Based on test data :  2.8396178363003424


In [25]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('MAE : ', mean_absolute_error(y_test, y_pred))
print('MSE : ', mean_squared_error(y_test, y_pred))
print('RMSE : ', np.sqrt(mean_squared_error(y_test, y_pred)))
print('R² Score : ', r2_score(y_test, y_pred))

MAE :  0.5006337789130235
MSE :  0.31646410468312314
RMSE :  0.5625514240343927
R² Score :  0.6988495493206516


In [26]:
import joblib

joblib.dump(model, "crop_yield_model.pkl")
joblib.dump(scaler, "crop_yield_scaler.pkl")
joblib.dump(encoder, "crop_yield_encoder.pkl")

print("Files saved successfully!")

Files saved successfully!
